In [1]:
!pip install pytubefix

In [2]:
!pip install llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu121

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu121


In [3]:
 !pip install -q gradio yt-dlp torch transformers sentencepiece protobuf accelerate faiss-cpu openai-whisper pytubefix


In [ ]:
# %% [markdown]
# # Arabic Audio Intelligence - Gradio on Google Colab (GPU)
# **Setup:** Runtime → Change runtime type → T4 GPU

# %% [code] — Cell 1: Install dependencies

# %% [code] — Cell 2: Main app
import os
import tempfile
import numpy as np
import torch
import gradio as gr
from pathlib import Path
import uuid
import subprocess
from pytubefix import YouTube

# ---------- device ----------
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🏎️ Running on {DEVICE.upper()}")

# ---------- model holders ----------
whisper_model    = None
summarizer_model = None
bert_tokenizer   = None
bert_model       = None

# ---------- model loaders ----------
def load_whisper():
    global whisper_model
    if whisper_model is None:
        import whisper
        print("Loading Whisper large-v3...")
        whisper_model = whisper.load_model("small", device=DEVICE)
        print("✅ Whisper loaded")
    return whisper_model

def load_summarizer():
    global summarizer_model, whisper_model
    if summarizer_model is None:
        # Free Whisper from VRAM first
        if whisper_model is not None:
            del whisper_model
            whisper_model = None
            torch.cuda.empty_cache()
            print("🗑️ Whisper unloaded to free VRAM")

        from huggingface_hub import hf_hub_download
        from llama_cpp import Llama
        print("Downloading Mistral-7B-Q4 (~4GB, one time)...")
        model_path = hf_hub_download(
            repo_id="bartowski/Mistral-7B-Instruct-v0.3-GGUF",
            filename="Mistral-7B-Instruct-v0.3-Q4_K_M.gguf",
        )
        print("Loading Mistral into GPU...")
        summarizer_model = Llama(
            model_path=model_path,
            n_ctx=4096,        # reduced from 8192 to save VRAM
            n_gpu_layers=-1,
            verbose=False,
        )
        print("✅ Mistral loaded")
    return summarizer_model

def load_embedder():
    global bert_tokenizer, bert_model
    if bert_model is None:
        from transformers import AutoTokenizer, AutoModel
        print("Loading multilingual-e5-large...")
        name = "intfloat/multilingual-e5-large"
        bert_tokenizer = AutoTokenizer.from_pretrained(name)
        bert_model     = AutoModel.from_pretrained(name).to(DEVICE)
        bert_model.eval()
        print("✅ Embedder loaded")
    return bert_tokenizer, bert_model

# ---------- helpers ----------
def seconds_to_srt(s):
    h, m   = int(s // 3600), int((s % 3600) // 60)
    sc, ms = int(s % 60), int((s - int(s)) * 1000)
    return f"{h:02d}:{m:02d}:{sc:02d},{ms:03d}"

def seconds_to_ts(s):
    return f"{int(s//60):02d}:{int(s%60):02d}"

def generate_srt(segments):
    lines = []
    for i, seg in enumerate(segments, 1):
        lines.append(str(i))
        lines.append(f"{seconds_to_srt(seg['start'])} --> {seconds_to_srt(seg['end'])}")
        lines.append(seg['text'])
        lines.append("")
    return "\n".join(lines)

# ---------- embeddings & search ----------
def get_embeddings(texts, prefix="passage: "):
    tok, mdl = load_embedder()
    prefixed = [prefix + t for t in texts]
    all_embs = []
    for i in range(0, len(prefixed), 16):
        batch = prefixed[i:i+16]
        enc   = tok(batch, padding=True, truncation=True, max_length=512, return_tensors="pt").to(DEVICE)
        with torch.no_grad():
            out = mdl(**enc)
        embs = out.last_hidden_state.mean(dim=1).cpu().numpy().astype(np.float32)
        all_embs.append(embs)
    embs  = np.vstack(all_embs)
    norms = np.linalg.norm(embs, axis=1, keepdims=True)
    return embs / np.where(norms == 0, 1, norms)

def build_search_index(segments):
    import faiss
    texts = [s["text"] for s in segments]
    embs  = get_embeddings(texts, prefix="passage: ")
    index = faiss.IndexFlatIP(embs.shape[1])
    index.add(embs)
    return index

def search_segments(query, segments, index, top_k=5):
    q_emb       = get_embeddings([query], prefix="query: ")
    scores, idx = index.search(q_emb, top_k)
    out = []
    for sc, i in zip(scores[0], idx[0]):
        if i < len(segments):
            out.append({"score": float(sc), **segments[i]})
    return out

# ---------- transcription ----------
def transcribe_audio(path):
    model = load_whisper()
    res   = model.transcribe(path, language="ar", verbose=False)
    full  = res["text"].strip()
    segs  = [
        {"start": s["start"], "end": s["end"], "text": s["text"].strip()}
        for s in res["segments"] if s["text"].strip()
    ]
    return segs, full

# ---------- summarization ----------
def summarize_text(text: str) -> str:
    mdl = load_summarizer()

    # Truncate to fit in context window (keep ~2500 words to leave room for prompt+output)
    words = text.split()
    if len(words) > 2500:
        text = " ".join(words[:2500])

    prompt = (
        "[INST] أنت مساعد متخصص في تلخيص النصوص العربية. "
        "لخص النص التالي تلخيصاً شاملاً ودقيقاً في فقرتين كحد أقصى، "
        "مع الحفاظ على أهم النقاط والأفكار الرئيسية. "
        "يجب أن يكون ردك باللغة العربية فقط وليس بأي لغة أخرى:\n\n"
        f"{text} [/INST]"
    )
    out = mdl(prompt, max_tokens=512, temperature=0.3, echo=False)
    return out["choices"][0]["text"].strip()

# ---------- YouTube download via pytubefix (Corrected) ----------
def download_youtube_audio(url: str):
    from pytubefix import YouTube
    tmp_dir = tempfile.mkdtemp()
    yt = YouTube(url)  # no client argument
    audio_stream = yt.streams.filter(only_audio=True).order_by("abr").desc().first()
    if not audio_stream:
        raise RuntimeError("No audio stream found.")
    out_file = audio_stream.download(output_path=tmp_dir, filename="audio")
    base, ext = os.path.splitext(out_file)
    new_file = base + '.mp3'
    os.rename(out_file, new_file)
    return new_file, yt.title

# ---------- global state ----------
current = {
    "segments":        None,
    "full_transcript": None,
    "summary":         None,
    "srt":             None,
    "index":           None,
}

# ---------- main pipeline ----------
def process(youtube_url, audio_file):
    global current
    audio_path    = None
    is_downloaded = False
    try:
        if youtube_url and youtube_url.strip():
            audio_path, title = download_youtube_audio(youtube_url.strip())
            is_downloaded = True
            status = f"✅ Downloaded: {title}"
        elif audio_file is not None:
            audio_path = audio_file
            title      = Path(audio_file).stem
            status     = f"📁 Uploaded: {title}"
        else:
            return "❌ Provide a URL or upload a file", "", None, None, None, "No input"

        status += "\n🎤 Transcribing with Whisper large-v3..."
        segs, full = transcribe_audio(audio_path)
        status += f"\n✔️ Transcription: {len(segs)} segments, {len(full.split())} words"

        status += "\n📝 Summarizing with Mistral-7B..."
        summary = summarize_text(full) if full.strip() else "لا يوجد نص"
        status += "\n✔️ Summary done"

        status += "\n🔍 Building search index with E5-large..."
        index       = build_search_index(segs)
        srt_content = generate_srt(segs)
        status += "\n✔️ Index ready — all done!"

        current = {
            "segments":        segs,
            "full_transcript": full,
            "summary":         summary,
            "srt":             srt_content,
            "index":           index,
        }

        html = '<div style="max-height:500px;overflow:auto;">'
        for s in segs:
            html += f'''
            <div style="display:flex;gap:12px;padding:8px;margin-bottom:6px;
                        background:#1e1e2a;border-radius:10px;">
                <span style="font-family:monospace;color:#c9a84c;
                             min-width:90px;flex-shrink:0;">
                    {seconds_to_ts(s["start"])} → {seconds_to_ts(s["end"])}
                </span>
                <span style="direction:rtl;text-align:right;flex:1;">
                    {s["text"]}
                </span>
            </div>'''
        html += '</div>'

        uid      = uuid.uuid4().hex[:8]
        srt_path = f"/tmp/subtitles_{uid}.srt"
        txt_path = f"/tmp/transcript_{uid}.txt"
        sum_path = f"/tmp/summary_{uid}.txt"

        with open(srt_path, "w", encoding="utf-8") as f: f.write(srt_content)
        with open(txt_path, "w", encoding="utf-8") as f: f.write(full)
        with open(sum_path, "w", encoding="utf-8") as f: f.write(summary)

        if is_downloaded and audio_path and os.path.exists(audio_path):
            os.remove(audio_path)

        return html, summary, srt_path, txt_path, sum_path, status

    except Exception as e:
        import traceback
        err = traceback.format_exc()
        print(err)
        return "", "", None, None, None, f"❌ Error:\n{err}"

# ---------- search ----------
def search(query):
    if current["index"] is None:
        return "<div style='color:#f43f5e;padding:1rem;'>⚠️ Process an audio first.</div>"
    if not query or not query.strip():
        return "<div style='color:#6b7280;padding:1rem;'>🔍 Enter a query above.</div>"

    res = search_segments(query.strip(), current["segments"], current["index"], top_k=5)
    if not res:
        return "<div style='color:#6b7280;padding:1rem;'>No matching segments.</div>"

    out = "<div style='padding:.5rem 0'><strong style='color:#8b5cf6'>Top results</strong> <span style='color:#6b7280;font-size:.8rem'>(cosine similarity)</span></div>"
    for r in res:
        pct       = int(max(0, min(100, r["score"] * 100)))
        bar_color = "#8b5cf6" if pct > 70 else "#c9a84c" if pct > 50 else "#6b7280"
        out += f'''
        <div style="background:#1e1e2a;padding:12px 16px;border-radius:10px;
                    margin-top:10px;border-left:3px solid {bar_color};">
            <div style="display:flex;justify-content:space-between;
                        font-family:monospace;font-size:.75rem;margin-bottom:6px;">
                <span style="color:#c9a84c;">⏱ {seconds_to_ts(r["start"])} → {seconds_to_ts(r["end"])}</span>
                <span style="color:{bar_color};">Score: {pct}%</span>
            </div>
            <div style="background:#0a0a0f;height:3px;border-radius:4px;margin-bottom:8px;">
                <div style="width:{pct}%;height:100%;background:{bar_color};border-radius:4px;"></div>
            </div>
            <div style="direction:rtl;text-align:right;line-height:1.6;">{r["text"]}</div>
        </div>'''
    return out

# ---------- Gradio UI ----------
css = """
@import url('https://fonts.googleapis.com/css2?family=Tajawal:wght@300;400;700;900&family=JetBrains+Mono:wght@400;600&display=swap');
body, .gradio-container { background:#0a0a0f !important; color:#f0ede6 !important; font-family:'Tajawal',sans-serif !important; }
.gradio-container { max-width:960px !important; margin:auto !important; }
h1 { text-align:center; background:linear-gradient(135deg,#f0ede6,#c9a84c,#8b5cf6);
     -webkit-background-clip:text; background-clip:text; color:transparent;
     font-size:2rem; font-weight:900; margin-bottom:.25rem; }
.gr-button-primary   { background:linear-gradient(135deg,#c9a84c,#a07830) !important;
                       color:#0a0a0f !important; font-weight:700 !important; border:none !important; }
.gr-button-secondary { background:#1c1c28 !important; color:#8b5cf6 !important;
                       border:1px solid #8b5cf6 !important; }
input, textarea { background:#1c1c28 !important; border:1px solid #2a2a3a !important;
                  color:#f0ede6 !important; font-family:'Tajawal',sans-serif !important; }
label { color:#c9a84c !important; font-size:.8rem !important; letter-spacing:.1em !important; }
"""

with gr.Blocks(title="Arabic Audio Intelligence", css=css) as demo:
    gr.HTML("""
        <h1>🎙️ Arabic Audio Intelligence</h1>
        <p style='text-align:center;color:#6b7280;font-size:.9rem;margin-bottom:1.5rem;'>
            Whisper large-v3 · Mistral-7B · E5-large — تحويل الصوت إلى نص · التلخيص · البحث الدلالي
        </p>
    """)

    with gr.Row():
        with gr.Column(scale=2):
            url_in  = gr.Textbox(label="YouTube URL", placeholder="https://www.youtube.com/watch?v=...")
            file_in = gr.File(label="Or upload audio (mp3 / wav / m4a)", file_types=[".mp3", ".wav", ".m4a"])
            go_btn  = gr.Button("🚀 Process", variant="primary")
        with gr.Column(scale=1):
            status_out = gr.Textbox(label="Status", lines=8, interactive=False)

    with gr.Tabs():
        with gr.TabItem("📜 Transcript"):
            transcript_html = gr.HTML()
            with gr.Row():
                srt_dl = gr.File(label="⬇️ Download SRT")
                txt_dl = gr.File(label="⬇️ Download Full Text")
        with gr.TabItem("📄 Summary"):
            summary_text = gr.Textbox(label="Summary (Mistral-7B)", lines=10, interactive=False)
            sum_dl = gr.File(label="⬇️ Download Summary")
        with gr.TabItem("🔎 Search"):
            search_q   = gr.Textbox(label="Search query", placeholder="ما هو الموضوع الرئيسي؟")
            search_btn = gr.Button("🔍 Search", variant="secondary")
            search_out = gr.HTML()

    go_btn.click(
        fn=process,
        inputs=[url_in, file_in],
        outputs=[transcript_html, summary_text, srt_dl, txt_dl, sum_dl, status_out],
    )
    search_btn.click(
        fn=search,
        inputs=[search_q],
        outputs=[search_out],
    )

demo.launch(share=True, debug=True)

🏎️ Running on CUDA


/tmp/ipykernel_22481/2383804038.py:301: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(title="Arabic Audio Intelligence", css=css) as demo:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://7df2d48ab3f70db9d9.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 421, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 56, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1159, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

Loading Whisper large-v3...
✅ Whisper loaded


...

100%|██████████| 15397/15397 [00:22<00:00, 681.94frames/s]


Loading multilingual-e5-large...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedder loaded
